<a href="https://colab.research.google.com/github/Zinc-zn/MachineLearning2026/blob/main/JS09/JS09_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# JS09 - Approximate Nearest Neighbors: Lab 4 (ANNOY vs. FAISS vs. HNSW)

Percobaan membandingkan ketiga model ANN yang telah dibahas pada **1.000.000 data berdimensi 5**. Perhatikan perbedaan waktu *build index* dan waktu *query* dari masing-masing algoritma.

> **Catatan:** sel eksekusi ini memerlukan waktu beberapa menit (pembangunan index Annoy dan HNSW pada 1 juta data).


In [1]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# Build 1mio rows with 5D
# ===============================
n_data = 1_000_000   # try 100_000 if you have limited memory
dim = 5
np.random.seed(42)
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")

=== Annoy ===


Build time: 20.62480854988098 detik
Query time: 0.0010073184967041016 detik
Neighbors: [901095, 495168, 561017, 163986, 185734] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.013001680374145508 detik
Query time: 0.011483430862426758 detik
Neighbors: [901095 495168 561017 163986 185734] ...

=== HNSW (hnswlib) ===


Build time: 123.20344066619873 detik
Query time: 0.0 detik
Neighbors: [901095 495168 561017 163986 185734] ...


**Analisis hasil (waktu pasti berbeda antar mesin):**

* **Annoy** memiliki *build time* paling lama di antara yang approximate (membangun 10 pohon secara berulang memproses seluruh 1 juta item via loop Python), namun *query time*-nya sangat cepat.
* **FAISS IndexFlatL2** *build*-nya hampir instan (hanya menyalin data), tetapi query-nya *exact* brute force sehingga paling lambat saat query pada data besar.
* **HNSW** *build time* paling lama (membangun graf hierarki), namun menghasilkan query tercepat sekaligus hasil yang (pada kasus ini) sama dengan exact.
* Ketiga algoritma menemukan 5 tetangga teratas yang sama pada percobaan ini - pada data random berdimensi rendah, index ANN cenderung sangat akurat.